# Практика. NER как генерация — дообучение и промптинг

Вы узнали, как использовать современные языковые модели к задаче выделения именованных сущностей. В этом уроке вы на практике сравните плюсы и минусы разных подходов: возьмёте одну задачу, подготовите данные в разных форматах, оцените время работы методов и их метрики качества.

### Установка зависимостей
Подготовьте необходимое библиотеки. Помимо стандартного набора, нам понадобится VLLM. 

In [2]:
!pip install transformers==4.55.2 torch==2.8.0 numpy==1.26.1 datasets==4.0.0 accelerate==1.10.0 evaluate==0.4.6 vllm==0.10.2 seqeval==1.2.2

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of cupy-cuda12x to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 59.6 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.0/888.0 MB 18.1 MB/s  0:00:19m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 54.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.4/436.4 MB 24.8 MB/s  0:00:14m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 39.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 24.9 MB/s  0:00:13m0:00:010

## Загрузка данных
Для экспериментов мы будем использовать данные из [RuMedBench](https://arxiv.org/pdf/2201.06499), а именно — часть бенчмарка по задаче NER. Он состоит из размеченных предложений на тему приёма лекарств. Предложения взяты из отзывов на лекарства.

### Задание 1
1. Загрузите [архив с файлами датасета](https://code.s3.yandex.net/deep-learning/medicine_dataset.zip).
1. Создайте переменную `ner_dataset` класса `datasets.DatasetDict`, в которой будут храниться все загруженные сплиты: `train`, `dev` (aka validation) и `test`.
1. Создайте переменную `labels_list` — список со всеми тегами, которые есть в обучающей выборке.
Выполните задание на ВМ, выданной в предыдущем уроке. Затем сверьтесь с авторским решением. 

In [4]:
from datasets import Dataset, DatasetDict, load_dataset

data_files = {
    "train": "medicine_dataset/train_v1.jsonl.txt",
    "val": "medicine_dataset/dev_v1.jsonl.txt",
    "test": "medicine_dataset/test_v1.jsonl.txt"
}

ner_dataset = load_dataset("json", data_files=data_files)
ner_dataset

DatasetDict({
    train: Dataset({
        features: ['idx', 'tokens', 'ner_tags'],
        num_rows: 3440
    })
    val: Dataset({
        features: ['idx', 'tokens', 'ner_tags'],
        num_rows: 676
    })
    test: Dataset({
        features: ['idx', 'tokens', 'ner_tags'],
        num_rows: 693
    })
})

In [5]:
train_tags = [tag for tags in list(ner_dataset["train"]["ner_tags"]) for tag in tags]
val_tags = [tag for tags in list(ner_dataset["val"]["ner_tags"]) for tag in tags]
test_tags = [tag for tags in list(ner_dataset["test"]["ner_tags"]) for tag in tags]

labels_list = list(set(train_tags + val_tags + test_tags))
labels_list.sort()
labels_list

['B-ADR',
 'B-DI',
 'B-Drugclass',
 'B-Drugform',
 'B-Drugname',
 'B-Finding',
 'I-ADR',
 'I-DI',
 'I-Drugclass',
 'I-Drugform',
 'I-Drugname',
 'I-Finding',
 'O']

Мы будем работать с шестью классами: ADR, DI, Drugname, Drugform, Drugclass, Finding. ADR — побочные реакции, DI — взаимодействие лекарственных препаратов.

Посмотрим на примеры:

In [6]:
from IPython.display import HTML, display
import html

def visualize_tokens(tokens, tags):
    """
    tokens: список токенов
    tags: BIO-теги для этих токенов
    """

    html_output = ""
    i = 0
    while i < len(tokens):
        tag = tags[i]

        if tag.startswith("B-"):  # начало сущности
            ent_type = tag[2:]
            ent_tokens = [tokens[i]]

            # собираем все I- токены этого же типа
            j = i + 1
            while j < len(tokens) and tags[j] == f"I-{ent_type}":
                ent_tokens.append(tokens[j])
                j += 1

            # объединяем в один span
            ent_text = " ".join(ent_tokens)
            html_output += f"<span style='background-color: #ffd54f; padding:2px; margin:1px; border-radius:4px;'>{html.escape(ent_text)} <sub>{ent_type}</sub></span> "
            i = j
        else:
            # токен вне сущности
            html_output += html.escape(tokens[i]) + " "
            i += 1

    display(HTML(html_output))


for i in range(3):
    ex = ner_dataset['train'][i]
    print('\nПример', i)
    visualize_tokens(ex['tokens'], ex['ner_tags'])


Пример 0



Пример 1



Пример 2


Ожидаемый вывод — строки, в которых в оранжевых боксах выделены сущности. Каждой бокс подписан соответствующим классом.

В датасете примеры хранятся в виде списка слов, каждому слову сопоставлен тег.

## Метрики
Подготовим функцию для оценки качества на основе библиотеки `seqeval`. У функций `seqeval` есть параметры, которые помогают настроить их поведение. Нам подойдёт стандартные значения, нужно только предварительно преобразовать предсказанные индексы классов в строковые теги.

### Задание 2

Допишите функцию для оценки метрик во время обучения `compute_metrics_trainer`. Используйте готовую `convert_preds_to_labels`. Верните словарь с `precision`, `recall` и `f1-score`.

In [7]:
import numpy as np
from typing import Any, List, Tuple, Dict
import evaluate


seqeval = evaluate.load('seqeval')

def convert_preds_to_labels(predictions: np.ndarray, label_ids: np.ndarray, label_list: List[str]) -> Tuple[List[List[str]], List[List[str]]]:
    """
    Преобразует predictions (логиты) и истинные label_ids в списки меток в строковом виде для seqeval.
    Ожидается: predictions.shape = (batch, seq_len, num_labels) или (batch, seq_len) если уже argmax.
    label_ids — числовые метки с -100 для игнорируемых токенов.
    """
    if predictions.ndim == 3:
        preds = np.argmax(predictions, axis=-1)
    else:
        preds = predictions
    true_labels = []
    pred_labels = []
    for pred_row, label_row in zip(preds, label_ids):
        tl_row = []
        pl_row = []
        for p, l in zip(pred_row, label_row):
            # игнорируем паддинг
            if p == -100 or l == -100:
                continue
            tl_row.append(label_list[l])
            pl_row.append(label_list[p])
        true_labels.append(tl_row)
        pred_labels.append(pl_row)
    return pred_labels, true_labels

def compute_metrics_trainer(eval_pred: Any) -> Dict[str, Any]:
    """Функция для Trainer.compute_metrics — возвращает dict с результатами seqeval."""
    predictions, label_ids = eval_pred
    pred_labels, true_labels = convert_preds_to_labels(predictions, label_ids, labels_list)
    result = seqeval.compute(predictions=pred_labels, references=true_labels)

    return {
        'precision': result.get('precision', None) or result.get('overall_precision'),
        'recall': result.get('recall', None) or result.get('overall_recall'),
        'f1': result.get('f1', None) or result.get('overall_f1'),
    }

## Донастройка BERT
В качестве бейзлайна используем BERT, обученный на задаче потокенной классификации.

Сначала подготовим данные в нужном формате. В датасете они хранятся в виде списка слов и соответствующих тегов. При этом токенизатор может дополнительно разбить слова на токены ещё меньшего размера. Нужно привести список тегов в соответствие последовательности токенов, которую выдаёт токенизатор.

In [8]:
from transformers import AutoTokenizer
model_name = 'bert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize_and_align_labels(batch, tokenizer, labels_list):
    # batch: dict с 'tokens' и 'ner_tags' (каждый — список примеров)
    tokenized_inputs = tokenizer(batch['tokens'],
                                 is_split_into_words=True,
                                 truncation=True,
                                 padding='max_length',
                                 max_length=128,
                                 return_tensors=None)
    all_labels = []
    for i, labels in enumerate(batch['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels_list.index(labels[word_idx]))
            else:
                if labels[word_idx] == "O":
                    label_ids.append(labels_list.index(labels[word_idx]))
                else:
                    label_ids.append(labels_list.index("I-"+labels[word_idx].split("-")[1]))
            previous_word_idx = word_idx
        all_labels.append(label_ids)
    tokenized_inputs['labels'] = all_labels
    return tokenized_inputs


tokenized_ds = ner_dataset.map(tokenize_and_align_labels, batched=True, fn_kwargs={"tokenizer": tokenizer, "labels_list": labels_list})
tokenized_ds

DatasetDict({
    train: Dataset({
        features: ['idx', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3440
    })
    val: Dataset({
        features: ['idx', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 676
    })
    test: Dataset({
        features: ['idx', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 693
    })
})

Теперь подготовим объект класса `Trainer`, который отвечает за всё обучение модели. Для этого сначала определимся с гиперпараметрами и загрузим веса модели.


### Задание 3
Допишите аргументы, необходимые для обучения. Если нужно, обратитесь к подсказке. 

In [9]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

num_labels = len(labels_list)
model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=num_labels)

args = TrainingArguments(
    output_dir='bert-ner',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to="none",
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Обучим наш бейзлайн. Обучение займёт несколько минут. 

In [49]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["val"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics_trainer,
)
trainer.train()

Epoch,Training Loss,Validation Loss


/home/ubuntu/project/.venv/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


NameError: name 'result' is not defined

## Инференс
Для инференса удобно создать пайплайн (`transformers.pipeline`). Он совместит в себе предобработку входной строки и постобработку предсказанных тегов. На выходе мы получим сгруппированные теги с указанием на подстроку.

In [1]:
from transformers import pipeline
model.config.label2id = {l: i for i, l in enumerate(labels_list)}
model.config.id2label = {i: l for i, l in enumerate(labels_list)}
p = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="simple")


sent = " ".join(ner_dataset['test'][0]["tokens"])
print(sent)
p(sent)

/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 7/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

Чтобы визуализировать результаты и подсчитать метрики, приведём вывод к формату датасета — списку тегов, соответствующих словам.

In [16]:
from typing import List, Dict, Tuple

def convert_pipeline_output_to_bio(pipeline_output: List[Dict], tokens: List[str], label_list: List[str]) -> List[str]:
    tags = ["O"] * len(tokens)
    token_char_indices = []
    current_char_idx = 0
    for token in tokens:
        token_char_indices.append((current_char_idx, current_char_idx + len(token)))
        current_char_idx += len(token) + 1 # Account for space

    for entity in pipeline_output:
        start_char = entity['start']
        end_char = entity['end']
        entity_type = entity['entity_group']

        start_token_idx = -1
        end_token_idx = -1

        # Find the token indices corresponding to the entity character span
        for i, (token_start_char, token_end_char) in enumerate(token_char_indices):
            # Check if the entity span starts within or at the beginning of the token span
            if start_char >= token_start_char and start_char < token_end_char:
                 start_token_idx = i
            # Check if the entity span ends within or at the end of the token span
            if end_char > token_start_char and end_char <= token_end_char:
                 end_token_idx = i

        if start_token_idx != -1 and end_token_idx != -1:
            tags[start_token_idx] = f"B-{entity_type}"
            for i in range(start_token_idx + 1, end_token_idx + 1):
                tags[i] = f"I-{entity_type}"

    return tags

In [ ]:
test_sentence_tokens = ner_dataset['test'][0]["tokens"]
test_sent = " ".join(test_sentence_tokens)
pipeline_result = p(test_sent)
bio_tags = convert_pipeline_output_to_bio(pipeline_result, test_sentence_tokens, labels_list)
print("Tokens:", test_sentence_tokens)
print("BIO Tags:", bio_tags)

visualize_tokens(test_sentence_tokens, bio_tags)

Видно, что модель справляется не идеально: помимо сущностей, относящихся к лекарствам, выделяет общие слова, не несущие важного смысла. К примеру, слово «название», вопреки разметке модели, не указывает на взаимодействие лекарственных препаратов.

In [ ]:
eval_bert_finetune_metrics = trainer.evaluate(tokenized_ds['test'])
eval_bert_finetune_metrics

В результате обучения в течение 3 эпох вы должны получить  f1-score > 0.4. Это неплохой результат для модели маленького размера, не специализированной в медицине.

## LLM + prompt
Задачу NER можно решить и с помощью языковых моделей. Для этого опишем в инструкции, какие элементы требуется извлечь из сырого текста. Возьмём модель Qwen3-1.7B.

### Задание 4
Реализуйте инференс обученной языковой модели с помощью библиотеки `transformers`. Используйте токенизатор, подготовьте сообщения для входа в LLM и декодируйте её выход.

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re
slug = "Qwen/Qwen3-0.6B"
qwen_tokenizer = AutoTokenizer.from_pretrained(slug)
qwen_model = AutoModelForCausalLM.from_pretrained(slug, device_map="auto")


example_id = 1
example_sentence = " ".join(tokenized_ds['train'][example_id]["tokens"])
example_tokens = tokenized_ds['train'][example_id]["tokens"]
example_tags = tokenized_ds["train"][example_id]["ner_tags"]


@torch.no_grad
def inference(model, tokenizer, sentence):
    tokens = [tokenizer.decode(tkn) for tkn in tokenizer.encode(sentence)]
    prompt = f"""<task>
Проведи задачу NER - распознавание именованных сущностей. Присвой BIO-метку каждому токену предложения ниже.
Метки: ['B-ADR', 'B-DI', 'B-Drugclass', 'B-Drugform', 'B-Drugname', 'B-Finding', 'I-ADR', 'I-DI', 'I-Drugclass', 'I-Drugform', 'I-Drugname', 'I-Finding', 'O']
Значение префиксов меток:
- "B-" - токен начала сущности
- "I-" - токен продолжения сущности
- "O" - токен не относящийся ни к одной из сущностей
Значение меток:
- ADR (Adverse Drug Reaction) - неблагоприятная лекарственная реакция;
- DI (Drug Interaction) - лекарственное взаимодействие;
- Drugclass - класс лекарств;
- Drugform - лекарственная форма;
- Drugname - название лекарства;
- Finding - вывод, заключение или обнаружение в контексте медицинских исследований или диагностики.
</task>

<output_format>
Оформи ответ в виде JSON-списка BIO-меток. Только JSON, никаких других символов!!!
Пример ответа: ["O", "O", "O", "B-Drugform", "O", "O", "O", "O", "B-Drugclass"]
</output_format>

<example>
Предложение: {example_sentence}
Токены: {example_tokens}
BIO-метки: {example_tags}
</example>

<input>
Перечисли BIO-метки для токенов этого предложения:
Предложение: {sentence}
Токены: {tokens}
</input>
"""
    messages = [
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=32768
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    content = tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")

    return content

In [12]:
sentence = " ".join(tokenized_ds['test'][0]["tokens"])
raw_output = inference(qwen_model, qwen_tokenizer, sentence)
print(f"Sentence: {sentence}")
print(f"Tokens: {tokenized_ds['test'][0]['tokens']}")
print(f"Target: {tokenized_ds['test'][0]['ner_tags']}")
print(f"Predict: {raw_output}")

Sentence: Вылезла простуда на губах , заказала мужу , чтоб купил мазь Ацикловир , ну вообщем он забыл название , и купил , то , что собственно ему предложили в аптеке , прорекламировав так , что мол жена оценит некий матирующий эффект от данного средства .
Tokens: ['Вылезла', 'простуда', 'на', 'губах', ',', 'заказала', 'мужу', ',', 'чтоб', 'купил', 'мазь', 'Ацикловир', ',', 'ну', 'вообщем', 'он', 'забыл', 'название', ',', 'и', 'купил', ',', 'то', ',', 'что', 'собственно', 'ему', 'предложили', 'в', 'аптеке', ',', 'прорекламировав', 'так', ',', 'что', 'мол', 'жена', 'оценит', 'некий', 'матирующий', 'эффект', 'от', 'данного', 'средства', '.']
Target: ['O', 'B-DI', 'I-DI', 'I-DI', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Drugform', 'B-Drugname', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Predict: ["O", "B-Drugclass", "O", "O", "O", "O", "O", "O", "B-DI", "I-DI", "I-DI", "O", "O

Распарсим ответ LLM — извлечём список выделенных сущностей

In [14]:
import json

# def prompt_output_to_bio(tokens: List[str], prompt_output: str, label_list: List[str]) -> List[int]:
#     ner_tags = [label_list.index('O')] * len(tokens)
#     # Попробуем найти JSON
#     parsed = None
#     pattern = r"```json\n([\s\S]*?)\n```"
#     matches = re.findall(pattern, prompt_output)
#     try:
#         parsed = json.loads(matches[0])
#     except Exception:
#         # Попробуем найти шаблон 'entity: Type; entity2: Type'
#         parts = [p.strip() for p in prompt_output.replace(';', '\n').split('\n') if p.strip()]
#         parsed = []
#         for p in parts:
#             if ':' in p:
#                 left, right = p.split(':', 1)
#                 parsed.append({'text': left.strip().strip('"'), 'type': right.strip()})

#     # parsed ожидается как список объектов {'text':..., 'type':...}
#     if isinstance(parsed, list):
#         for ent in parsed:
#             text = ent.get('text') if isinstance(ent, dict) else None
#             typ = ent.get('type') if isinstance(ent, dict) else None
#             if not text or not typ:
#                 continue
#             # простая стратегия: ищем последовательность токенов, равную text.split()
#             ent_toks = text.split()
#             # naive search
#             for i in range(len(tokens) - len(ent_toks) + 1):
#                 window = tokens[i:i+len(ent_toks)]
#                 if [w.lower().strip('.,') for w in window] == [w.lower().strip('.,') for w in ent_toks]:
#                     b_label = f'B-{typ}'
#                     i_label = f'I-{typ}'
#                     if b_label in label_list:
#                         ner_tags[i] = label_list.index(b_label)
#                         for j in range(1, len(ent_toks)):
#                             ner_tags[i+j] = label_list.index(i_label) if i_label in label_list else ner_tags[i+j]
#                     break
#     return ner_tags

def prompt_output_to_bio(tokens: List[str], prompt_output: str, label_list: List[str]) -> List[int]:
    try:
        ner_tags = json.loads(prompt_output)
    except:
        ner_tags = [label_list.index('O')] * len(tokens)
    return ner_tags

Теперь можно использовать заданные ранее способы визуализации

In [15]:
test_id = 1
test_sentence = " ".join(tokenized_ds['train'][test_id]["tokens"])
test_tokens = tokenized_ds['train'][test_id]["tokens"]
test_tags = tokenized_ds["train"][test_id]["ner_tags"]


raw_output = inference(qwen_model, qwen_tokenizer, test_sentence)
print('Выход LLM:', raw_output)

bio_tags = prompt_output_to_bio(test_tokens, raw_output, labels_list)
print('Сконвертированные теги:', bio_tags)

visualize_tokens(test_tokens, bio_tags)

Выход LLM: ["O", "B-Drugname", "O", "O", "O", "O", "O", "O", "B-DI", "I-DI", "I-DI", "O", "O", "O", "O", "O", "O", "O", "O", "O"]
Сконвертированные теги: ['O', 'B-Drugname', 'O', 'O', 'O', 'O', 'O', 'O', 'B-DI', 'I-DI', 'I-DI', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [16]:
test_id = 0
test_sentence = " ".join(tokenized_ds['train'][test_id]["tokens"])
test_tokens = tokenized_ds['train'][test_id]["tokens"]
test_tags = tokenized_ds["train"][test_id]["ner_tags"]


raw_output = inference(qwen_model, qwen_tokenizer, test_sentence)
print('Выход LLM:', raw_output)

bio_tags = prompt_output_to_bio(test_tokens, raw_output, labels_list)
print('Сконвертированные теги:', bio_tags)

visualize_tokens(test_tokens, bio_tags)

Выход LLM: ["O", "B-Drugname", "O", "O", "O", "O", "O", "B-Drugform", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O"]
Сконвертированные теги: ['O', 'B-Drugname', 'O', 'O', 'O', 'O', 'O', 'B-Drugform', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


Чтобы полноценно замерить этот метод, нужно много времени, поэтому сразу улучшим его.

## VLLM + Structured Outputs
Чтобы ускорить работу LLM, можно использовать библиотеку для быстрого инференса, например, vLLM. В ней же можно ограничить формат выхода.

Сначала опишем схему ожидаемого результата с помощью `Pydantic`.

### Задание 5
Допишите реализацию классов, чтобы получить описание результата.

In [18]:
import pydantic
from typing import List
from enum import Enum


class EntityType(Enum):
    adr = "ADR"
    di = "DI"
    drugclass = "Drugclass"
    drugform = "Drugform"
    drugname = "Drugname"
    finding = "Finding"

class Entity(pydantic.BaseModel):
    text: str
    type: EntityType

class Result(pydantic.BaseModel):
    entities: List[Entity]

Теперь подготовим схему вместе с остальными `SamplingParams`:

In [19]:
from vllm import SamplingParams
from vllm.sampling_params import GuidedDecodingParams

json_schema = Result.model_json_schema()
guided = GuidedDecodingParams(json=json_schema)
sampling_params = SamplingParams(guided_decoding=guided, max_tokens=500)

Наконец, инициализируем сам инстанс с LLM, а также вспомогательные функции для его использования:

In [20]:
from vllm import LLM
import torch
torch.cuda.empty_cache()


slug = "Qwen/Qwen3-1.7B"
llm = LLM(
    model=slug,
    guided_decoding_backend='xgrammar',
    max_num_batched_tokens=512,
    max_model_len=4096,
    gpu_memory_utilization=0.6,
)


def schema_inference(model, sentence, sampling_params):
    prompt = 'Extract entities of class ADR (adverse drug reaction), DI (drug interference), Drugclass, Drugform, Drugname or Finding from the sentence. Return as JSON list of {"text": quote_from_text, "type": assigned_class}. Skip non-mentioned classes\n' + sentence
    outputs = llm.generate(prompts=prompt, sampling_params=sampling_params, use_tqdm=False)
    return Result.model_validate_json(outputs[0].outputs[0].text).entities

INFO 05-30 20:45:30 [__init__.py:216] Automatically detected platform cuda.


INFO 05-30 20:45:31 [utils.py:328] non-default args: {'max_model_len': 4096, 'gpu_memory_utilization': 0.6, 'max_num_batched_tokens': 512, 'disable_log_stats': True, 'guided_decoding_backend': 'xgrammar', 'model': 'Qwen/Qwen3-1.7B'}
INFO 05-30 20:45:42 [__init__.py:742] Resolved architecture: Qwen3ForCausalLM
WARNING 05-30 20:45:42 [__init__.py:2716] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 05-30 20:45:42 [__init__.py:2767] Casting torch.bfloat16 to torch.float16.
INFO 05-30 20:45:42 [__init__.py:1815] Using max model len 4096
INFO 05-30 20:45:44 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=512.
WARNING 05-30 20:45:45 [__init__.py:2974] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUD

[W530 20:45:55.882910161 ProcessGroupNCCL.cpp:981] Warning: TORCH_NCCL_AVOID_RECORD_STREAMS is the default now, this environment variable is thus deprecated. (function operator())


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=6867) INFO 05-30 20:45:55 [parallel_state.py:1165] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=6867) WARNING 05-30 20:45:55 [topk_topp_sampler.py:69] FlashInfer is not available. Falling back to the PyTorch-native implementation of top-p & top-k sampling. For the best performance, please install FlashInfer.
(EngineCore_DP0 pid=6867) INFO 05-30 20:45:55 [gpu_model_runner.py:2

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.23it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.23it/s]
(EngineCore_DP0 pid=6867) 


(EngineCore_DP0 pid=6867) INFO 05-30 20:45:58 [default_loader.py:268] Loading weights took 1.71 seconds
(EngineCore_DP0 pid=6867) INFO 05-30 20:45:59 [gpu_model_runner.py:2392] Model loading took 3.2152 GiB and 2.717695 seconds
(EngineCore_DP0 pid=6867) INFO 05-30 20:46:05 [backends.py:539] Using cache directory: /home/ubuntu/.cache/vllm/torch_compile_cache/0265e96acf/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=6867) INFO 05-30 20:46:05 [backends.py:550] Dynamo bytecode transform time: 6.24 s
(EngineCore_DP0 pid=6867) INFO 05-30 20:46:08 [backends.py:161] Directly load the compiled graph(s) for dynamic shape from the cache, took 2.674 s
(EngineCore_DP0 pid=6867) INFO 05-30 20:46:09 [monitor.py:34] torch.compile takes 6.24 s in total
(EngineCore_DP0 pid=6867) INFO 05-30 20:46:10 [gpu_worker.py:298] Available KV cache memory: 4.13 GiB
(EngineCore_DP0 pid=6867) INFO 05-30 20:46:11 [kv_cache_utils.py:864] GPU KV cache size: 38,672 tokens
(EngineCore_DP0 pid=6867) INFO 05

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:03<00:00, 17.68it/s]


(EngineCore_DP0 pid=6867) INFO 05-30 20:46:15 [gpu_model_runner.py:3118] Graph capturing finished in 4 secs, took 1.71 GiB
(EngineCore_DP0 pid=6867) INFO 05-30 20:46:15 [gpu_worker.py:391] Free memory on device (11.47/14.58 GiB) on startup. Desired GPU memory utilization is (0.6, 8.75 GiB). Actual usage is 3.22 GiB for weight, 1.39 GiB for peak activation, 0.01 GiB for non-torch memory, and 1.71 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=2446768742` to fit into requested memory, or `--kv-cache-memory=5365257216` to fully utilize gpu memory. Current kv cache memory in use is 4436965990 bytes.
(EngineCore_DP0 pid=6867) INFO 05-30 20:46:15 [core.py:218] init engine (profile, create kv cache, warmup model) took 16.42 seconds
INFO 05-30 20:46:16 [llm.py:295] Supported_tasks: ['generate']
INFO 05-30 20:46:16 [__init__.py:36] No IOProcessor plugins requested by the model


In [21]:
from tqdm.auto import tqdm

def schema_to_bio(tokens, parsed, label_list):
    ner_tags = [label_list.index('O')] * len(tokens)
    # parsed - список объектов класса Entity
    for ent in parsed:
        text = ent.text
        typ = str(ent.type)
        if not text or not typ:
            continue
        # простая стратегия: ищем последовательность токенов в предложении, равную text.split()
        ent_toks = text.split()
        # 
        for i in range(len(tokens) - len(ent_toks) + 1):
            window = tokens[i:i+len(ent_toks)]
            if [w.lower().strip('.,') for w in window] == [w.lower().strip('.,') for w in ent_toks]:
                b_label = f'B-{typ}'
                i_label = f'I-{typ}'
                if b_label in label_list: # проверяем, что такой тег действительно есть в нашей задаче, иначе мы не знаем, с чем его сравнивать.
                    ner_tags[i] = label_list.index(b_label)  # сначала добавим в список тегов открывающийся тег
                    for j in range(1, len(ent_toks)):
                        ner_tags[i+j] = label_list.index(i_label) if i_label in label_list else ner_tags[i+j] # а затем - все внутренние теги
                break
    return ner_tags


def evaluate_llm_schema_on_dataset(model, dataset, label_list, sampling_params):
    true_labels = []
    pred_labels = []

    for example in tqdm(dataset):
        tokens = example['tokens']
        true_tags = example['ner_tags']

        sentence = " ".join(tokens)
        try:
          raw_output = schema_inference(model, sentence, sampling_params=sampling_params)
        except pydantic.ValidationError: 
          # если что-то пошло не по плану - например, генерация зациклилась, то распарсить ответ не получится - пропустим такие примеры
          raw_output = dict()

        pred_tags = schema_to_bio(tokens, raw_output, label_list)

        true_labels.append(true_tags)
        pred_labels.append([label_list[tag] for tag in pred_tags])


    results = seqeval.compute(predictions=pred_labels, references=true_labels)
    return results

In [22]:
num_samples = 10
eval_llm_schema_metrics = evaluate_llm_schema_on_dataset(llm, ner_dataset['test'].select(range(num_samples)), labels_list, sampling_params)
eval_llm_schema_metrics

  0%|          | 0/10 [00:00<?, ?it/s]

/home/ubuntu/project/.venv/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/ubuntu/project/.venv/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'DI': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 4},
 'Drugform': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3},
 'Drugname': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2},
 'overall_precision': 0.0,
 'overall_recall': 0.0,
 'overall_f1': 0.0,
 'overall_accuracy': 0.9395604395604396}

Ожидаемый результат: f1-score = 0.25. Это достаточно низкое значение, но оно получено на небольшой языковой модели без дообучения — и в этом контексте оно неплохое!

Заметим, что VLLM работает значительно быстрее, чем HF, а также даёт дополнительную возможность управлять форматом выхода модели.

## Дообучение в формате T5

Последний подход, который мы применим, — это дообучение модели T5. От бейзлайна этот подход отличается генеративным взглядом на извлечение сущностей — прямо как в предыдущих способах с языковыми моделями. Вместо T5 для обучения можно использовать и Qwen3 из предыдущего подхода (потребуется лишь немного поменять формат входных данных). 

### Задание 6
Отформатируем датасет: приведём список тегов к нужному виду — список словарей, в каждом из них ключ — это цитата из текста, а значение — её тип.  Для этого допишите функцию `make_target_from_entities`, которая получает на вход список токенов и соответствующих тегов, а возвращает строку, в которой описан словарь `"фраза": "тип"`.

Выполните задание на выданной ВМ. Затем сверьтесь с авторским решением. Для разнообразия в нём используется `mT5-small` от Google, так что при желании можете использовать эту модель и в своём решении. 

In [23]:
from typing import List, Tuple
import json

def extract_entities_intervals(tags: List[str]) -> List[Tuple[str, int, int]]:
    """
    Преобразует список тегов (в виде индексов) в интервалы сущностей.
    Возвращает список кортежей: (label, start_idx, end_idx).
    """
    entities = []
    start, end, ent_type = None, None, None

    for i, label in enumerate(tags):

        if label == "O":
            if ent_type is not None:
                entities.append((ent_type, start, end))
                ent_type, start, end = None, None, None
        elif label.startswith("B-"):
            if ent_type is not None:
                entities.append((ent_type, start, end))
            ent_type = label[2:]
            start, end = i, i
        elif label.startswith("I-") and ent_type == label[2:]:
            end = i
        else:
            if ent_type is not None:
                entities.append((ent_type, start, end))
            ent_type, start, end = None, None, None

    if ent_type is not None:
        entities.append((ent_type, start, end))

    return entities


def make_target_from_entities(tokens: List[str], tags: List[str]) -> str:
    ents = extract_entities_intervals(tags)
    output = dict()
    for ent_type, start, end in ents:
        key = " ".join(tokens[start:end+1])
        output[key] = ent_type
    return json.dumps(output, ensure_ascii=False)

In [24]:
test_id = 0
test_sentence = " ".join(tokenized_ds['train'][test_id]["tokens"])
test_tokens = tokenized_ds['train'][test_id]["tokens"]
test_tags = tokenized_ds["train"][test_id]["ner_tags"]

make_target_from_entities(test_tokens, test_tags)

'{"Орвирем": "Drugname"}'

Применим преобразование к датасету и токенизируем входные и выходные строки.

In [25]:
from datasets import Dataset, DatasetDict

def prepare_seq2seq_dataset(dataset):
    # функция, которая преобразует каждый сплит
    def _convert_split(split):
        records = []
        for ex in dataset[split]:
            input_text = " ".join(ex["tokens"]) # склеенное входное предложение
            target_text = make_target_from_entities(ex["tokens"], ex["ner_tags"]) # строка с ожидаемым результатом генерации
            records.append({"input_text": input_text, "target_text": target_text})
        return Dataset.from_list(records)

    out = {}
    for split in dataset.keys():
        out[split] = _convert_split(split)
    return DatasetDict(out)


def tokenize_seq2seq(batch):
    # токенизация входной строки
    model_inputs = tokenizer(batch["input_text"], padding="max_length", truncation=True, max_length=256)
    # токенизация выхода
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(batch["target_text"], padding="max_length", truncation=True, max_length=256)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


seq2seq_dataset = prepare_seq2seq_dataset(ner_dataset)
tokenized_dataset = seq2seq_dataset.map(tokenize_seq2seq, batched=True)

print(seq2seq_dataset['train'][0])

Map:   0%|          | 0/3440 [00:00<?, ? examples/s]

/home/ubuntu/project/.venv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:4006: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/676 [00:00<?, ? examples/s]

Map:   0%|          | 0/693 [00:00<?, ? examples/s]

{'input_text': 'Орвирем как многие лекарственные средства , нам выписала врач педиатр .', 'target_text': '{"Орвирем": "Drugname"}'}


Загрузим модель.

Дообучим её на нашей задаче. Для этого инициализируем уже знакомые нам переменные и классы.

In [29]:
from peft import get_peft_model, PromptTuningConfig, TaskType
from typing import List
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer,\
    Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq


# model_name = 'google/mt5-small'
model_name = "google/flan-t5-small"


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [36]:
import torch
torch.cuda.empty_cache()


args = Seq2SeqTrainingArguments(
    output_dir="./t5_ner",
    eval_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    predict_with_generate=True,
    logging_dir="./logs",
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["val"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.026850
2,0.052300,0.022770
3,0.036000,0.022021


TrainOutput(global_step=1290, training_loss=0.041506030208380645, metrics={'train_runtime': 445.5177, 'train_samples_per_second': 23.164, 'train_steps_per_second': 2.896, 'total_flos': 959195004272640.0, 'train_loss': 0.041506030208380645, 'epoch': 3.0})

Оценим дообученную модель. Подготовим функции по аналогии с предыдущими подходами.

### Задача 7
Реализуйте функцию, которая будет конвертировать текст-выход T5 в последовательность тегов в исходном формате датасета. Ориентируйтесь на функции для предыдущих подходов.

In [ ]:
def t5_output_to_bio(decoded_text, tokens, label_list):
    ner_tags = ["O"] * len(tokens)
    entities = {}

    try:
        entities = eval(decoded_text)
    except:
        decoded_text = decoded_text.replace("{", "").replace("}", "")
        parts = [p.strip() for p in decoded_text.split(',') if p.strip()]
        for p in parts:
            if ':' in p:
                text, typ = p.split(':', 1)
                entities[text.strip()] = typ.strip()

    for entity_text, entity_type in entities.items():
        ent_toks = entity_text.split()
        for i in range(len(tokens) - len(ent_toks) + 1):
            window = tokens[i:i+len(ent_toks)]

            if [w.lower().strip('.,') for w in window] == [w.lower().strip('.,') for w in ent_toks]:
                b_label = f"B-{entity_type}"
                i_label = f"I-{entity_type}"
                if b_label in label_list:
                    ner_tags[i] = b_label
                    for j in range(1, len(ent_toks)):
                        if i + j < len(tokens):
                            ner_tags[i+j] = i_label if i_label in label_list else ner_tags[i+j]
                break
    return ner_tags


raw_t5_output = '{"мазь": "Drugform"}'
tags = t5_output_to_bio(raw_t5_output, test_tokens, labels_list)
print("Сконвертированные теги: ", tags) 
# Сконвертированные теги:  ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Drugform', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Сконвертированные теги:  ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


Подготовим технические функции для оценки метрик:

In [ ]:
import numpy as np
import torch


def evaluate_t5_with_seqeval(trainer, dataset, raw_dataset, label_list):
    predictions = trainer.predict(dataset, max_length=256)

    preds = np.where(predictions.predictions != -100, predictions.predictions, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True, )
    decoded_labels = tokenizer.batch_decode(predictions.label_ids, skip_special_tokens=True)

    true_labels = []
    pred_labels = []

    for i in range(len(dataset)):
        original_tokens = raw_dataset[i]["tokens"]
        true_tags = raw_dataset[i]["ner_tags"]
        true_labels.append(true_tags)

        pred_bio_tags = t5_output_to_bio(decoded_preds[i], original_tokens, labels_list)
        pred_labels.append(pred_bio_tags)

    t5_seqeval_results = seqeval.compute(predictions=pred_labels, references=true_labels)
    return t5_seqeval_results


@torch.no_grad
def t5_inference(sentence: str, model, tokenizer) -> List[str]:
    input_ids = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True, max_length=256).input_ids.to(model.device)
    generated_ids = model.generate(input_ids, max_new_tokens=256)
    decoded_output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    return decoded_output

Визуализируем пример. У недообученной модели, к примеру, могут быть нежелательные зацикливания.



In [52]:
test_id = 0
test_sentence = " ".join(tokenized_ds['train'][test_id]["tokens"])
test_tokens = tokenized_ds['train'][test_id]["tokens"]
test_tags = tokenized_ds["train"][test_id]["ner_tags"]


raw_t5_output = t5_inference(test_sentence, model, tokenizer)
tags = t5_output_to_bio(raw_t5_output, test_tokens, labels_list)
print("Выход T5: ", raw_t5_output)
print("Сконвертированные теги: ", tags) 

visualize_tokens(test_tokens, tags)

Выход T5:  WeIh as many specialized industries , we had been working on the project .
Сконвертированные теги:  ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


И, наконец, замерим метрики.

In [54]:
num_samples = 10  # если есть время, проведите замер на полном тестовом датасете 
eval_t5_results = evaluate_t5_with_seqeval(trainer, tokenized_dataset['test'].select(range(num_samples)), ner_dataset['test'].select(range(num_samples)), labels_list)
eval_t5_results

/home/ubuntu/project/.venv/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/ubuntu/project/.venv/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'DI': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 4},
 'Drugform': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3},
 'Drugname': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2},
 'overall_precision': 0.0,
 'overall_recall': 0.0,
 'overall_f1': 0.0,
 'overall_accuracy': 0.9395604395604396}

Ожидаемое значение после обучения в течение 3 эпох: f1-score = 0.24. Это достаточно низкий результат, который коррелирует с зацикливаниями, которые вы видели выше. Значение метрики обусловлено размером модели и тем фактом, что входные данные отличаются от данных, использованных во время обучения. Попробуйте продолжить обучение и посмотреть, повысится ли качество.